In [1]:
import json
from pathlib import Path

In [2]:
with open('./data/target_api_resp.json') as f:
    api_resp = json.load(f)

In [3]:
# from resp
targets = []
for resp in api_resp:
    targets = [*targets, *resp.get('targets', [])]

In [4]:
# from db
#targets = []
with open('./data/kpf_cc_targets_2025A.txt') as f:
    targets = [*targets , *json.load(f)]

In [5]:
targets[-1]

{'_id': '678fc0740287e8cb84fbba90',
 'del_flag': 0,
 'details': 'Passed all infeasibility checks',
 'id': '7accb434-c230-5603-94b8-9bf210354dce',
 'last_modification': '2025-01-21 05:43:31',
 'num_intranight_cadence': 0,
 'num_visits_per_night': 1,
 'obsid': 5808,
 'semid': '2025A_N056',
 'simulcal_on': True,
 'state': 'TARGET_SUBMITTED',
 'status': 'SUCCESS',
 'submitter': 'Rafael Luque',
 'target_feasible': True,
 'need_resubmit': False,
 'target_name': 'K2-155',
 'dec': '+21:21:12.937879152',
 'dec_deg': 21.353593855,
 'epoch': 'J2000',
 'g_mag': 12.241566,
 'gaia_id': 'DR3_145333927996558976',
 'j_mag': 10.274,
 'pm_dec': -76.775,
 'pm_ra': 200.181,
 'ra': '04:21:52.4852745864',
 'ra_deg': 65.4686886441,
 'systemic_velocity': 18.94271,
 'tic_id': 'No_TIC_Name',
 't_eff': 4258,
 'maximum_exposure_time': 600,
 'nominal_exposure_time': 600,
 'num_exposures_per_visit': 1,
 'num_unique_nights_per_semester': 5,
 'num_internight_cadence': 1,
 'days_observable': 179,
 'isNew': False,
 'ris

In [7]:
obs = []
for cc_target in targets:
    target = {
    	'target_name': (cc_target.get('target_name', 'undefined')),
    	'gaia_id': cc_target.get('gaia_id'),
        'tic_id': cc_target.get('tic_id'),
    	'systemic_velocity': cc_target.get('systemic_velocity'),
    	'g_mag': cc_target.get('g_mag'),
    	'j_mag': cc_target.get('j_mag'),
    	't_eff': cc_target.get('t_eff'),
    	'ra': cc_target.get('ra'),
    	'dec': cc_target.get('dec'),
        'ra_deg': cc_target.get('ra_deg'),
        'dec_deg': cc_target.get('dec_deg'),
    	'pm_ra': cc_target.get('pm_ra'),
    	'pm_dec': cc_target.get('pm_dec'),
    	'epoch': cc_target.get('epoch')
        }
    observation = {
        'exposure_time': cc_target.get('nominal_exposure_time'),
        'num_exposures': cc_target.get('num_exposures_per_visit'),
        }
    
    schedule = {
    	'scheduling_mode': 'Cadence',
    	'num_visits_per_night': cc_target.get('num_visits_per_night'),
    	'num_nights_per_semester': cc_target.get('num_unique_nights_per_semester'),
    	'num_internight_cadence': cc_target.get('num_internight_cadence'),
    	'num_intranight_cadence': cc_target.get('num_intranight_cadence'),
        'total_observations_requested': cc_target.get('total_observations_requested'),
        'total_time_for_target': cc_target.get('total_time_for_target'),
        'total_time_for_target_hours': cc_target.get('total_time_for_target_hours')
        }
    
    metadata = {
    	'obsid': cc_target.get('obsid'),
    	'observer_name': cc_target.get('submitter'),
    	'semester': cc_target.get('semid').split('_')[0],
    	'progid': cc_target.get('semid').split('_')[1],
    	'semid': cc_target.get('semid'),
    	'history': [],
        'submitter': cc_target.get('submitter'),
    	'tags': ['imported from legacy targets'],
         #'details': cc_target.get('details'),
         #'status': cc_target.get('status'),
        'comment': cc_target.get('comment'),
        }
    
    ob = {
        'del_flag': 0,
        'metadata': metadata,
    	'target': target,
    	'observation': observation,
    	'schedule': schedule,
    	'calibration': {}
        }
    obs.append(ob)


In [8]:
with open('legacy_obs_2025_04_17.json', 'w') as f:
    json.dump(obs, f, indent=4)

# transfer instructions

1. sudo mongo observing_dev --quiet --eval 'db.kpf_cc_targets.find({semid: {$regex: '2025A_'}, del_flag: 0})' > /kpf_cc_2025A.json
2. scp file over to where this script is run
3. run script (you may need to sub out ' with " and put quotes around the keywords
4. scp legacy_obs.json dsibld@10.136.1.80:/home/dsibld/legacy_obs_2425.json  
5. head over to vm-odb2 as dsibld
6. Optional: run db.kpf_cc_observing_block_dev.deleteMany({"metadata.tags": {$in: ['imported from legacy targets']}, "metadata.semester": {$in: ["2024A", "2024B", "2025A"]}})
7. mongoimport --db kpf_cc --collection kpf_cc_observing_block_dev --jsonArray --file legacy_obs.json

In [6]:
componentNames = ['metadata', 'target', 'observation', 'schedule', 'calibration']
schemaFiles = ['metadata_schema.json', 'ob_target_schema.json', 'observation_schema.json', 'schedule_data_schema.json', 'calibration_schema.json']
def get_schema_dict():
    schemas = {}
    schemaPath = Path('./schemas/')
    for component, file in zip(componentNames, schemaFiles):
        with open(schemaPath / file) as f:
            schema = json.load(f)
        schemas[component] = schema
    return schemas

inverse_map = lambda myMap: {v: k for k, v in myMap.items()}

def make_schema_mapping():
    obSchema = get_schema_dict()
    obKeyMapping = {}
    for ckey, schema in obSchema.items():
        properties = schema['properties']
        componentMap = {}
        for key, props in properties.items():
            componentMap[props.get('translator_mapping', key)] = key
        obKeyMapping[ckey] = componentMap
    return obKeyMapping

def swap_translator_ob_to_ob_keys(ob, keyMap):
    # converts ob to use KPF Translator keys
    OB = {}
    if '_id' in ob.keys():
        OB['_id'] = ob['_id']
    OB['del_flag'] = ob['del_flag']
    components = [[key, comp] for key, comp in ob.items() if key in componentNames]
    for ckey, component in components:
        Component = {}
        for key, value in component.items():
            Component[keyMap[ckey][key]] = value
        OB[ckey] = Component
    return OB

In [7]:
OBobKeyMapping = make_schema_mapping()
obOBKeyMapping = {key: inverse_map(mp) for key, mp in OBobKeyMapping.items()}

In [8]:
OBS = []
for ob in obs:
    OB = swap_translator_ob_to_ob_keys(ob, obOBKeyMapping)
    OBS.append(OB)